# 🌾 KrishiMitra – Train the model on Google Colab

**Before you start:** `Runtime → Change runtime type → T4 GPU`, then run the cells from top to bottom.

What this notebook does:
1. Gets your code from GitHub and the datasets from Kaggle / GitHub
2. Creates stratified train/val/test splits
3. Trains EfficientNet-B0 (and optionally MobileNetV3) with transfer learning
4. Evaluates on PlantVillage **and** on real field photos (PlantDoc)
5. Creates a Grad-CAM heatmap, exports to ONNX and saves everything to Google Drive

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Google Drive (keeps your files safe when Colab restarts)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/krishimitra'
import os; os.makedirs(f'{DRIVE}/zips', exist_ok=True); os.makedirs(f'{DRIVE}/artifacts', exist_ok=True)

## 2. Get the code  ✏️ change `YOUR_USERNAME`

In [ ]:
REPO_URL = 'https://github.com/YOUR_USERNAME/krishimitra.git'
!rm -rf /content/krishimitra && git clone -q $REPO_URL /content/krishimitra
%cd /content/krishimitra
!pip install -q timm==1.0.11 albumentations==1.4.21 grad-cam==1.5.4 onnx==1.17.0 onnxruntime==1.20.1 python-dotenv pyyaml kaggle

## 3. Download datasets
Upload your `kaggle.json` (Kaggle → Settings → API → Create New Token) when asked. The ZIP is saved to Drive, so this is slow only the first time.

In [ ]:
import os
from pathlib import Path
from google.colab import files

if not Path('~/.kaggle/kaggle.json').expanduser().exists():
    files.upload()  # choose kaggle.json
    !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

zip_path = f'{DRIVE}/zips/plantvillage-dataset.zip'
if not os.path.exists(zip_path):
    !kaggle datasets download -d abdallahalidev/plantvillage-dataset -p {DRIVE}/zips

# Unzip only the colour images to fast local disk
!unzip -q -o "{zip_path}" "*/color/*" -d /content/pv
color_dir = next(p for p in Path('/content/pv').rglob('color') if p.is_dir())
Path('data/raw/plantvillage').mkdir(parents=True, exist_ok=True)
if not Path('data/raw/plantvillage/color').exists():
    os.symlink(color_dir, 'data/raw/plantvillage/color')

# PlantDoc – real field photos for the honest robustness test
!git clone -q --depth 1 https://github.com/pratikkayal/PlantDoc-Dataset.git data/raw/plantdoc

classes = sorted(d.name for d in color_dir.iterdir() if d.is_dir())
print('PlantVillage classes:', len(classes), '| images:', sum(len(os.listdir(color_dir / c)) for c in classes))

## 4. Create train / val / test splits (70 / 15 / 15, stratified)

In [ ]:
!python -m src.data.dataset

## 5. Train EfficientNet-B0  (~30–45 min on a T4)

In [ ]:
!python -m src.models.train --arch efficientnet_b0

### (Optional) Train MobileNetV3 to compare – smaller and faster

In [ ]:
# !python -m src.models.train --arch mobilenetv3_large_100

## 6. Evaluate
First the PlantVillage test set, then **real field photos** (PlantDoc). Expect PlantDoc accuracy to be much lower – report it honestly, it is a great viva talking point.

In [ ]:
!python -m src.models.evaluate --checkpoint models/efficientnet_b0_best.pth
!python -m src.models.evaluate --checkpoint models/efficientnet_b0_best.pth --external-dir data/raw/plantdoc/test --name plantdoc

In [ ]:
import json
for name in ['test', 'plantdoc']:
    m = json.load(open(f'reports/efficientnet_b0_{name}_metrics.json'))
    print(f"{name:9s} acc={m['accuracy']:.3f}  macroF1={m['macro_f1']:.3f}  top3={m['top3_accuracy']:.3f}  n={m['num_images']}  latency={m['cpu_latency_ms']}ms")

In [ ]:
from IPython.display import Image as Show
Show('reports/efficientnet_b0_test_confusion.png', width=900)

## 7. Grad-CAM – where does the model look?

In [ ]:
import glob
sample = sorted(glob.glob('data/raw/plantvillage/color/Tomato___Late_blight/*'))[0]
!python -m src.models.gradcam --checkpoint models/efficientnet_b0_best.pth --image "{sample}" --out reports/gradcam_example.jpg
Show('reports/gradcam_example.jpg', width=300)

## 8. Export to ONNX (for the server) and save everything to Drive

In [ ]:
!python -m src.models.export --checkpoint models/efficientnet_b0_best.pth
!cp models/krishimitra.onnx models/krishimitra.int8.onnx models/classes.json models/*_best.pth {DRIVE}/artifacts/
!cp -r reports {DRIVE}/artifacts/
!ls -lh {DRIVE}/artifacts

## ✅ Next steps
1. Download `krishimitra.onnx` and `classes.json` from Drive → `MyDrive/krishimitra/artifacts` into your local `models/` folder.
2. Run the app locally: `uvicorn api.main:app --reload` → http://127.0.0.1:8000
3. Copy the numbers from `reports/*_metrics.json` into `README.md` and `models/MODEL_CARD.md`.
4. Publish the model: `python scripts/upload_model.py --repo YOUR_USERNAME/krishimitra-model`